# Lab 3 — Exercise 1 (Toy manufacturing / sales prediction)

**Student**: 22521609 – Phạm Duy Tuấn  

## Dataset (Exercise 1)

We encode the table from the lab document as a list of records.  
**Target column**: `Sales` ∈ {`"High"`, `"Low"`}.  
**Features**: `Type`, `Colors` (number of colours), `Size`, `Material`.

In [1]:
import math
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score

FEATURES = ["Type", "Colors", "Size", "Material"]
LABEL = "Sales"

data = [
    {"Type": "Control", "Colors": 3, "Size": "Small", "Material": "PP Plastic", "Sales": "High"},
    {"Type": "Puzzle", "Colors": 5, "Size": "Medium", "Material": "Rubber", "Sales": "Low"},
    {"Type": "Puzzle", "Colors": 7, "Size": "Large", "Material": "PP Plastic", "Sales": "Low"},
    {"Type": "Control", "Colors": 5, "Size": "Small", "Material": "Rubber", "Sales": "Low"},
    {"Type": "Doll", "Colors": 3, "Size": "Medium", "Material": "PP Plastic", "Sales": "Low"},
    {"Type": "Control", "Colors": 5, "Size": "Medium", "Material": "PP Plastic", "Sales": "High"},
    {"Type": "Doll", "Colors": 5, "Size": "Large", "Material": "PP Plastic", "Sales": "High"},
    {"Type": "Control", "Colors": 7, "Size": "Medium", "Material": "Rubber", "Sales": "Low"},
    {"Type": "Puzzle", "Colors": 7, "Size": "Large", "Material": "Rubber", "Sales": "High"},
    {"Type": "Puzzle", "Colors": 3, "Size": "Large", "Material": "PP Plastic", "Sales": "Low"},
    {"Type": "Doll", "Colors": 3, "Size": "Small", "Material": "Rubber", "Sales": "Low"},
    {"Type": "Puzzle", "Colors": 3, "Size": "Small", "Material": "PP Plastic", "Sales": "High"},
    {"Type": "Control", "Colors": 5, "Size": "Large", "Material": "Rubber", "Sales": "Low"},
    {"Type": "Doll", "Colors": 5, "Size": "Medium", "Material": "PP Plastic", "Sales": "High"},
    {"Type": "Doll", "Colors": 7, "Size": "Large", "Material": "PP Plastic", "Sales": "High"},
]

df = pd.DataFrame(data)
print("Shape:", df.shape)
display(df)
print("\nClass distribution:")
display(df[LABEL].value_counts())

Shape: (15, 5)


,Type,Colors,Size,Material,Sales
0,Control,3,Small,PP Plastic,High
1,Puzzle,5,Medium,Rubber,Low
2,Puzzle,7,Large,PP Plastic,Low
3,Control,5,Small,Rubber,Low
4,Doll,3,Medium,PP Plastic,Low
5,Control,5,Medium,PP Plastic,High
6,Doll,5,Large,PP Plastic,High
7,Control,7,Medium,Rubber,Low
8,Puzzle,7,Large,Rubber,High
9,Puzzle,3,Large,PP Plastic,Low



Class distribution:


Sales
Low     8
High    7
Name: count, dtype: int64

## Question 1 — Information Gain (ID3)

**Lab task:** compute **information gain** for each attribute and build a **decision tree using ID3** (choose the split with **maximum** information gain at each node).

**Notes:**  
- **Entropy** \(H(S) = -\sum_i p_i \log_2 p_i\)  
- **Information gain** \(IG(S, A) = H(S) - \sum_v \frac{|S_v|}{|S|} H(S_v)\)

In [2]:
def entropy(labels):
    counter = Counter(labels)
    total = len(labels)
    if total == 0:
        return 0.0
    return -sum((c / total) * math.log2(c / total) for c in counter.values() if c > 0)


def info_gain(rows, attr, label_key=LABEL):
    """Information gain of splitting `rows` on categorical/numeric attribute `attr`."""
    labels = [r[label_key] for r in rows]
    total_entropy = entropy(labels)
    values = {r[attr] for r in rows}
    weighted = 0.0
    for v in values:
        subset = [r for r in rows if r[attr] == v]
        weighted += (len(subset) / len(rows)) * entropy([r[label_key] for r in subset])
    return total_entropy - weighted


print("Root entropy H(S):", round(entropy([r[LABEL] for r in data]), 6))
print("\nInformation gain per attribute (full dataset):")
for attr in FEATURES:
    ig = info_gain(data, attr)
    print(f"  {attr:12s}  IG = {ig:.6f}")


def id3(rows, attrs, label_key=LABEL):
    """Build ID3 tree: nested dict {attr: {value: subtree_or_leaf}} or a leaf label string."""
    labels = [r[label_key] for r in rows]
    if len(set(labels)) == 1:
        return labels[0]
    if not attrs:
        return Counter(labels).most_common(1)[0][0]

    gains = {a: info_gain(rows, a, label_key) for a in attrs}
    best = max(gains, key=gains.get)

    tree = {best: {}}
    for v in sorted({r[best] for r in rows}, key=lambda x: (str(type(x)), x)):
        subset = [r for r in rows if r[best] == v]
        if not subset:
            tree[best][v] = Counter(labels).most_common(1)[0][0]
        else:
            rest = [a for a in attrs if a != best]
            tree[best][v] = id3(subset, rest, label_key)
    return tree


tree_id3 = id3(data, list(FEATURES))
print("\nID3 tree (nested dict):")
tree_id3

Root entropy H(S): 0.996792

Information gain per attribute (full dataset):
  Type          IG = 0.025841
  Colors        IG = 0.006475
  Size          IG = 0.006475
  Material      IG = 0.185805

ID3 tree (nested dict):


{'Material': {'PP Plastic': {'Type': {'Control': 'High',
    'Doll': {'Colors': {3: 'Low', 5: 'High', 7: 'High'}},
    'Puzzle': {'Size': {'Large': 'Low', 'Small': 'High'}}}},
  'Rubber': {'Type': {'Control': 'Low',
    'Doll': 'Low',
    'Puzzle': {'Colors': {5: 'Low', 7: 'High'}}}}}}

## Question 2 — Gini index (CART)

**Lab task:** compute **weighted Gini** for each split and build a tree using **CART** idea: at each node pick the attribute with **lowest** weighted Gini (impurity after split).

**Gini** of labels: \(Gini = 1 - \sum_j p_j^2\).  
**Weighted Gini** for split on \(A\): \(\sum_v \frac{|S_v|}{|S|} Gini(S_v)\).

In [3]:
def gini(labels):
    counter = Counter(labels)
    total = len(labels)
    if total == 0:
        return 0.0
    return 1.0 - sum((c / total) ** 2 for c in counter.values())


def gini_split(rows, attr, label_key=LABEL):
    g = 0.0
    values = {r[attr] for r in rows}
    for v in values:
        subset = [r for r in rows if r[attr] == v]
        g += (len(subset) / len(rows)) * gini([r[label_key] for r in subset])
    return g


print("Weighted Gini after split (full dataset) — lower is better:")
for attr in FEATURES:
    gs = gini_split(data, attr)
    print(f"  {attr:12s}  weighted Gini = {gs:.6f}")


def cart(rows, attrs, label_key=LABEL):
    labels = [r[label_key] for r in rows]
    if len(set(labels)) == 1:
        return labels[0]
    if not attrs:
        return Counter(labels).most_common(1)[0][0]

    ginis = {a: gini_split(rows, a, label_key) for a in attrs}
    best = min(ginis, key=ginis.get)

    tree = {best: {}}
    for v in sorted({r[best] for r in rows}, key=lambda x: (str(type(x)), x)):
        subset = [r for r in rows if r[best] == v]
        if not subset:
            tree[best][v] = Counter(labels).most_common(1)[0][0]
        else:
            rest = [a for a in attrs if a != best]
            tree[best][v] = cart(subset, rest, label_key)
    return tree


tree_cart = cart(data, list(FEATURES))
print("\nCART-style tree (nested dict):")
tree_cart

Weighted Gini after split (full dataset) — lower is better:
  Type          weighted Gini = 0.480000
  Colors        weighted Gini = 0.493333
  Size          weighted Gini = 0.493333
  Material      weighted Gini = 0.377778

CART-style tree (nested dict):


{'Material': {'PP Plastic': {'Type': {'Control': 'High',
    'Doll': {'Colors': {3: 'Low', 5: 'High', 7: 'High'}},
    'Puzzle': {'Size': {'Large': 'Low', 'Small': 'High'}}}},
  'Rubber': {'Type': {'Control': 'Low',
    'Doll': 'Low',
    'Puzzle': {'Colors': {5: 'Low', 7: 'High'}}}}}}

## Shared helper — traverse a learned tree

The lab hint returns `"Unknown"` when a feature value was never seen under that branch during training.

In [4]:
def predict_tree(tree, sample):
    """Predict class label from a nested tree dict; samples omit the label key."""
    if not isinstance(tree, dict):
        return tree
    attr = next(iter(tree))
    value = sample[attr]
    if value not in tree[attr]:
        return "Unknown"
    return predict_tree(tree[attr][value], sample)


def pretty_tree(tree, indent=0):
    pad = "  " * indent
    if not isinstance(tree, dict):
        print(pad + f"=> {tree}")
        return
    for attr, branches in tree.items():
        print(pad + f"{attr}")
        for val, sub in branches.items():
            print(pad + f"  = {val!r}")
            pretty_tree(sub, indent + 2)


print("ID3 structure:")
pretty_tree(tree_id3)
print("\nCART structure:")
pretty_tree(tree_cart)

ID3 structure:
Material
  = 'PP Plastic'
    Type
      = 'Control'
        => High
      = 'Doll'
        Colors
          = 3
            => Low
          = 5
            => High
          = 7
            => High
      = 'Puzzle'
        Size
          = 'Large'
            => Low
          = 'Small'
            => High
  = 'Rubber'
    Type
      = 'Control'
        => Low
      = 'Doll'
        => Low
      = 'Puzzle'
        Colors
          = 5
            => Low
          = 7
            => High

CART structure:
Material
  = 'PP Plastic'
    Type
      = 'Control'
        => High
      = 'Doll'
        Colors
          = 3
            => Low
          = 5
            => High
          = 7
            => High
      = 'Puzzle'
        Size
          = 'Large'
            => Low
          = 'Small'
            => High
  = 'Rubber'
    Type
      = 'Control'
        => Low
      = 'Doll'
        => Low
      = 'Puzzle'
        Colors
          = 5
            => Low
          = 7
  

## Question 3 — Predict sales for new products (decision tree)

From the lab sheet:

| Type    | Colors | Size  | Material   |
|---------|-------|-------|------------|
| Doll    | 3     | Large | Rubber     |
| Puzzle  | 5     | Large | PP Plastic |
| Control | 3     | Large | Rubber     |

We show predictions for **both** ID3 and CART trees (the lab allows using *one* of them; we keep both for comparison in Q9).

In [5]:
test_samples_q3 = [
    {"Type": "Doll", "Colors": 3, "Size": "Large", "Material": "Rubber"},
    {"Type": "Puzzle", "Colors": 5, "Size": "Large", "Material": "PP Plastic"},
    {"Type": "Control", "Colors": 3, "Size": "Large", "Material": "Rubber"},
]

rows = []
for s in test_samples_q3:
    rows.append({
        **s,
        "pred_ID3": predict_tree(tree_id3, s),
        "pred_CART": predict_tree(tree_cart, s),
    })

pred_df = pd.DataFrame(rows)
display(pred_df)

y_pred_id3 = pred_df["pred_ID3"].tolist()
y_pred_cart = pred_df["pred_CART"].tolist()
print("ID3 predictions:", y_pred_id3)
print("CART predictions:", y_pred_cart)

,Type,Colors,Size,Material,pred_ID3,pred_CART
0,Doll,3,Large,Rubber,Low,Low
1,Puzzle,5,Large,PP Plastic,Low,Low
2,Control,3,Large,Rubber,Low,Low


ID3 predictions: ['Low', 'Low', 'Low']
CART predictions: ['Low', 'Low', 'Low']


## Question 4 — Confusion matrix, Accuracy, Recall

**Given actual sales:** `Low`, `Low`, `High` (in the same order as the three test cases above).

We evaluate **ID3** first (as a concrete choice; the lab says “results from tree”). If any prediction is `Unknown`, metrics may be ill-defined — we report that explicitly.

**Recall** is computed with `pos_label="High"` (positive class = High sales).

In [6]:
y_true_q4 = ["Low", "Low", "High"]

labels_order = ["High", "Low"]

if "Unknown" in y_pred_id3:
    print("ID3 encountered 'Unknown' on this tiny test set — sklearn metrics may be misleading.")
else:
    cm_id3 = confusion_matrix(y_true_q4, y_pred_id3, labels=labels_order)
    acc_id3 = accuracy_score(y_true_q4, y_pred_id3)
    rec_id3 = recall_score(y_true_q4, y_pred_id3, pos_label="High", zero_division=0)
    print("Confusion matrix (rows=true, cols=pred), labels=", labels_order)
    print(cm_id3)
    print("Accuracy:", acc_id3)
    print("Recall (positive class = High):", rec_id3)

# Also report CART for completeness
if "Unknown" not in y_pred_cart:
    cm_cart = confusion_matrix(y_true_q4, y_pred_cart, labels=labels_order)
    acc_cart = accuracy_score(y_true_q4, y_pred_cart)
    rec_cart = recall_score(y_true_q4, y_pred_cart, pos_label="High", zero_division=0)
    print("\nCART — confusion matrix:\n", cm_cart)
    print("CART Accuracy:", acc_cart)
    print("CART Recall (High):", rec_cart)

Confusion matrix (rows=true, cols=pred), labels= ['High', 'Low']
[[0 1]
 [0 2]]
Accuracy: 0.6666666666666666
Recall (positive class = High): 0.0

CART — confusion matrix:
 [[0 1]
 [0 2]]
CART Accuracy: 0.6666666666666666
CART Recall (High): 0.0


## Question 5 — Prior probability of `"Puzzle"`

Estimate \(\Pr(\text{Type}=\text{Puzzle})\) from the training table (maximum likelihood: counts / \(n\)).

In [7]:
count_puzzle = sum(1 for d in data if d["Type"] == "Puzzle")
prob_puzzle = count_puzzle / len(data)
print(f"P(Type = 'Puzzle') = {count_puzzle}/{len(data)} = {prob_puzzle:.6f}")

P(Type = 'Puzzle') = 5/15 = 0.333333


## Question 6 — Conditional probability \(\Pr(\text{Material}=\text{Rubber} \mid \text{Sales}=\text{Low})\)

Computed on the training rows where `Sales == "Low"`.

In [8]:
subset_low = [d for d in data if d[LABEL] == "Low"]
count_rubber_given_low = sum(1 for d in subset_low if d["Material"] == "Rubber")
prob_rubber_given_low = count_rubber_given_low / len(subset_low)
print(
    f"P(Material = 'Rubber' | Sales = 'Low') = "
    f"{count_rubber_given_low}/{len(subset_low)} = {prob_rubber_given_low:.6f}"
)

P(Material = 'Rubber' | Sales = 'Low') = 5/8 = 0.625000


## Question 7 — Naive Bayes + Laplace smoothing (same three samples as Q3)

We use **categorical Naive Bayes** with the lab’s Laplace smoothing for conditional probabilities:

\[
\hat P(x_j = v \mid y) = \frac{\text{count}(x_j=v, y)+1}{\text{count}(y)+|V_j|}
\]

where \(|V_j|\) is the number of **distinct** values of feature \(j\) in the full training data.

**Class prior** uses the same Laplace idea with 2 classes:

\[
\hat P(y) = \frac{\text{count}(y)+1}{n+2}.
\]

We compute scores in **log-space** for numerical stability: \(\log \hat P(y) + \sum_j \log \hat P(x_j\mid y)\).

In [9]:
label_counts = Counter(d[LABEL] for d in data)
CLASS_LABELS = ["High", "Low"]


def nb_cond_prob(attr, value, label):
    subset_y = [d for d in data if d[LABEL] == label]
    count = sum(1 for d in subset_y if d[attr] == value)
    k = len({d[attr] for d in data})
    return (count + 1) / (len(subset_y) + k)


def nb_prior(label):
    return (label_counts[label] + 1) / (len(data) + 2)


def predict_nb(sample):
    log_scores = {}
    for label in CLASS_LABELS:
        logp = math.log(nb_prior(label))
        for attr in FEATURES:
            logp += math.log(nb_cond_prob(attr, sample[attr], label))
        log_scores[label] = logp
    best = max(log_scores, key=log_scores.get)
    return best, log_scores


for i, s in enumerate(test_samples_q3, start=1):
    cls, scores = predict_nb(s)
    print(f"Sample {i} {s}")
    print(f"  log-scores: {scores}")
    print(f"  predicted Sales: {cls}\n")

y_pred_nb = [predict_nb(s)[0] for s in test_samples_q3]
print("Naive Bayes predictions (Q3):", y_pred_nb)

Sample 1 {'Type': 'Doll', 'Colors': 3, 'Size': 'Large', 'Material': 'Rubber'}
  log-scores: {'High': -5.2944034672269, 'Low': -4.469299197973208}
  predicted Sales: Low

Sample 2 {'Type': 'Puzzle', 'Colors': 5, 'Size': 'Large', 'Material': 'PP Plastic'}
  log-scores: {'High': -4.041640498731533, 'Low': -4.587082233629591}
  predicted Sales: High

Sample 3 {'Type': 'Control', 'Colors': 3, 'Size': 'Large', 'Material': 'Rubber'}
  log-scores: {'High': -5.5820855396786815, 'Low': -4.181617125521427}
  predicted Sales: Low

Naive Bayes predictions (Q3): ['Low', 'High', 'Low']


## Question 8 — Evaluate Naive Bayes on the three test cases

Same `y_true` as Q4: `["Low", "Low", "High"]`.

In [10]:
labels_order = ["High", "Low"]

cm_nb = confusion_matrix(y_true_q4, y_pred_nb, labels=labels_order)
acc_nb = accuracy_score(y_true_q4, y_pred_nb)
rec_nb = recall_score(y_true_q4, y_pred_nb, pos_label="High", zero_division=0)

print("Naive Bayes — confusion matrix (rows=true, cols=pred), labels=", labels_order)
print(cm_nb)
print("Accuracy:", acc_nb)
print("Recall (positive class = High):", rec_nb)

Naive Bayes — confusion matrix (rows=true, cols=pred), labels= ['High', 'Low']
[[0 1]
 [1 1]]
Accuracy: 0.3333333333333333
Recall (positive class = High): 0.0


## Question 9 — Compare Decision Tree vs Naive Bayes

On this **3-example** evaluation, we compare **accuracy** (and recall for `High`) between **ID3** and **NB**.  
This is a *tiny* test set, so differences should be interpreted cautiously.

In [11]:
if "Unknown" not in y_pred_id3:
    acc_tree = accuracy_score(y_true_q4, y_pred_id3)
    rec_tree = recall_score(y_true_q4, y_pred_id3, pos_label="High", zero_division=0)
else:
    acc_tree = float("nan")
    rec_tree = float("nan")
    print("ID3 had 'Unknown' predictions — accuracy/recall not reported.")

print(f"Decision Tree (ID3) Accuracy: {acc_tree}")
print(f"Naive Bayes Accuracy:         {acc_nb}")

if not math.isnan(acc_tree):
    print(f"\nDecision Tree (ID3) Recall (High): {rec_tree}")
    print(f"Naive Bayes Recall (High):         {rec_nb}")

    if acc_nb > acc_tree:
        print("\nOn this micro-test, Naive Bayes has higher accuracy.")
    elif acc_nb < acc_tree:
        print("\nOn this micro-test, ID3 has higher accuracy.")
    else:
        print("\nBoth models have the same accuracy on this micro-test.")

    print(
        "\nWhy they can differ: the tree partitions feature space with hard rules learned from "
        "co-occurrences in training, while Naive Bayes multiplies smoothed conditionals and assumes "
        "conditional independence given the class — on small data these inductive biases diverge."
    )

Decision Tree (ID3) Accuracy: 0.6666666666666666
Naive Bayes Accuracy:         0.3333333333333333

Decision Tree (ID3) Recall (High): 0.0
Naive Bayes Recall (High):         0.0

On this micro-test, ID3 has higher accuracy.

Why they can differ: the tree partitions feature space with hard rules learned from co-occurrences in training, while Naive Bayes multiplies smoothed conditionals and assumes conditional independence given the class — on small data these inductive biases diverge.


## Question 10 — New product

**Product:** Type = `Puzzle`, Colors = `7`, Size = `Small`, Material = `Rubber`  

(The lab PDF has a typo `"Smal"`; we use `"Small"`.)

Predict with **both** ID3 and Naive Bayes.

In [12]:
new_sample = {"Type": "Puzzle", "Colors": 7, "Size": "Small", "Material": "Rubber"}

print("Tree (ID3):", predict_tree(tree_id3, new_sample))
print("Tree (CART):", predict_tree(tree_cart, new_sample))
cls_nb, scores_nb = predict_nb(new_sample)
print("Naive Bayes:", cls_nb)
print("  log-scores:", scores_nb)

Tree (ID3): High
Tree (CART): High
Naive Bayes: Low
  log-scores: {'High': -5.869767612130463, 'Low': -4.756981270424989}
